# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HassanNawaz14/FlyRank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup — data, connection, and Week-4 baseline

*This section rebuilds `feature_df` from the warehouse and reproduces the Week-4 baseline rule exactly, so Section 3 has something honest to compare against. No new decisions are made here — everything below reuses validated logic from Weeks 3-4.*

In [ ]:
import os
os.makedirs('skills/directing-your-ai-assistant', exist_ok=True)
os.makedirs('skills/training-honest-models', exist_ok=True)
os.makedirs('skills/flyrank/flyrank-data', exist_ok=True)
!wget -q -O "skills/directing-your-ai-assistant/SKILL.md" "https://raw.githubusercontent.com/HassanNawaz14/FlyRank-ML-Internship/refs/heads/main/skills/directing-your-ai-assistant/SKILL.md"
!wget -q -O "skills/training-honest-models/SKILL.md" "https://raw.githubusercontent.com/HassanNawaz14/FlyRank-ML-Internship/refs/heads/main/skills/training-honest-models/SKILL.md"
!wget -q -O "skills/flyrank/flyrank-data/SKILL.md" "https://raw.githubusercontent.com/HassanNawaz14/FlyRank-ML-Internship/refs/heads/main/skills/flyrank/flyrank-data/SKILL.md"
print("skills loaded")


In [ ]:
import duckdb, pandas as pd, os

HF_TOKEN = os.getenv('HF_TOKEN')
if HF_TOKEN is None:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if HF_TOKEN is None:
    from getpass import getpass
    HF_TOKEN = getpass('Enter your Hugging Face token: ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}
print(con.execute(f"SELECT COUNT(*) FROM {TABLES['dim_clients']}").fetchone())


In [ ]:
# Materialize the expensive aggregation ONCE — fact_daily has ~79M rows
# spread across many remote parquet files. Do not repeat this step.
max_date = pd.to_datetime(con.execute(f"SELECT MAX(report_date) FROM {TABLES['fact_daily']}").fetchone()[0])
last30_start = max_date - pd.Timedelta(days=30)
prev30_start = max_date - pd.Timedelta(days=60)

con.execute(f"""
CREATE OR REPLACE TABLE agg_cached AS
SELECT client_hash_id, content_hash_id,
    SUM(CASE WHEN report_date > DATE '{last30_start.date()}' THEN gsc_impressions ELSE 0 END) AS imp_last30,
    SUM(CASE WHEN report_date > DATE '{last30_start.date()}' THEN gsc_clicks ELSE 0 END) AS clk_last30,
    AVG(CASE WHEN report_date > DATE '{last30_start.date()}' THEN gsc_avg_position END) AS pos_last30,
    SUM(CASE WHEN report_date > DATE '{prev30_start.date()}' AND report_date <= DATE '{last30_start.date()}' THEN gsc_impressions ELSE 0 END) AS imp_prev30,
    SUM(CASE WHEN report_date > DATE '{prev30_start.date()}' AND report_date <= DATE '{last30_start.date()}' THEN gsc_clicks ELSE 0 END) AS clk_prev30,
    AVG(CASE WHEN report_date > DATE '{prev30_start.date()}' AND report_date <= DATE '{last30_start.date()}' THEN gsc_avg_position END) AS pos_prev30
FROM {TABLES['fact_daily']}
GROUP BY client_hash_id, content_hash_id
""")
print(con.execute("SELECT COUNT(*) FROM agg_cached").fetchone())


In [ ]:
# dim_content's real columns (verified via DESCRIBE in week 3/4):
# client_hash_id, content_hash_id, keyword_hash_id, url_hash_id,
# keyword_char_count, keyword_token_count, url_char_count,
# content_created_date, content_updated_date, content_type, search_volume,
# competition, competition_level, cpc, main_intent, backlinks,
# category_count, keyword_created_date, provider_used, model_used,
# char_count, word_count, last_optimized_date, optimization_eligible_date,
# is_published, is_deleted. No days_since_last_update column -- derive it.

query = f"""
SELECT
    a.*,
    cl.access_profile, cl.gsc_data_start, cl.ga4_data_start,
    dc.word_count, dc.char_count, dc.content_type, dc.main_intent,
    dc.content_updated_date,
    DATE_DIFF('day', dc.content_updated_date, DATE '{max_date.date()}') AS days_since_last_update
FROM agg_cached a
JOIN {TABLES['dim_clients']} cl ON a.client_hash_id = cl.client_hash_id
LEFT JOIN {TABLES['dim_content']} dc ON a.content_hash_id = dc.content_hash_id
WHERE a.imp_prev30 >= 100
"""
feature_df = con.execute(query).fetchdf()
feature_df['ctr_prev30'] = feature_df['clk_prev30'] / feature_df['imp_prev30'] * 100
feature_df['is_declining_label'] = feature_df['imp_last30'] < 0.8 * feature_df['imp_prev30']
print(feature_df.shape)


In [ ]:
# Rebuild the Week-4 baseline rule EXACTLY as it was validated then,
# so this week's model has something real and honest to compare against.
import numpy as np

bins = [0, 90, 180, np.inf]
labels = ['<=90', '91-180', '181+']
feature_df['staleness_bucket'] = pd.cut(feature_df['days_since_last_update'], bins=bins, labels=labels, right=True)

pos_bins = [-np.inf, 3, 10, 20, 50, np.inf]
pos_labels = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']
feature_df['position_tier_prev'] = pd.cut(feature_df['pos_prev30'], bins=pos_bins, labels=pos_labels, right=True)

median_ctr_by_tier = feature_df.groupby('position_tier_prev', observed=True)['ctr_prev30'].median().rename('median_ctr_prev30').reset_index()
feature_df = feature_df.merge(median_ctr_by_tier, on='position_tier_prev', how='left')

feature_df['weak_ctr_flag'] = (
    (feature_df['ctr_prev30'] == 0) |
    ((feature_df['median_ctr_prev30'] > 0) & (feature_df['ctr_prev30'] < feature_df['median_ctr_prev30']))
)
feature_df['stale_flag'] = (feature_df['days_since_last_update'] > 90).astype(int)
feature_df['baseline_score'] = feature_df['stale_flag'] + feature_df['weak_ctr_flag'].astype(int)
feature_df['baseline_pred'] = (feature_df['baseline_score'] >= 1).astype(int)

print(feature_df['baseline_pred'].value_counts())


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Two models, not one.** Logistic Regression as the primary model — its coefficients show each signal's direction and magnitude directly, which matches this lane's actual question ("can I trust these signals") rather than just chasing accuracy. Random Forest with permutation importance as a second, nonlinear check — if a completely different method agrees with Logistic Regression on which signals matter most, that agreement is a much stronger finding than either model alone.

**Features used:** `imp_prev30`, `clk_prev30`, `pos_prev30`, `ctr_prev30`, `days_since_last_update`, `word_count`, `char_count`, `content_type`, `main_intent` — all pre-decision-window signals, safe per the leakage checks from Weeks 3-4.

**Explicitly excluded:** `imp_last30`, `clk_last30`, `pos_last30`, `trend_direction`, `trend_pct` — these overlap the window used to define `is_declining_label`, so using any of them as a feature would be direct leakage.

In [ ]:
feature_list = ['imp_prev30', 'clk_prev30', 'pos_prev30', 'ctr_prev30',
                'days_since_last_update', 'word_count', 'char_count',
                'content_type', 'main_intent']
print(feature_df[feature_list].isnull().sum())


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by `client_hash_id`, not random.** Pages from the same client tend to share templates and SEO practices — a random split would let the model see near-duplicate patterns from the same client in both train and test, making it look better than it actually is at generalizing to a genuinely new client. `GroupShuffleSplit` guarantees no client appears in both sides. The printed client-overlap count below must read 0 — if it doesn't, the split is not honest and needs fixing before training anything.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

feature_cols_numeric = ['imp_prev30', 'clk_prev30', 'pos_prev30', 'ctr_prev30',
                        'days_since_last_update', 'word_count', 'char_count']
feature_cols_categorical = ['content_type', 'main_intent']

model_df = feature_df.copy()
model_df[feature_cols_numeric] = model_df[feature_cols_numeric].fillna(model_df[feature_cols_numeric].median())
model_df[feature_cols_categorical] = model_df[feature_cols_categorical].fillna('unknown')

X = pd.get_dummies(model_df[feature_cols_numeric + feature_cols_categorical],
                    columns=feature_cols_categorical, drop_first=True)
y = model_df['is_declining_label'].astype(int)
groups = model_df['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
baseline_test = model_df.iloc[test_idx]['baseline_pred']

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
print(f"Train rows: {len(X_train)}, Test rows: {len(X_test)}")
print(f"Client overlap between train and test: {len(train_clients & test_clients)}")


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The code below trains Logistic Regression and Random Forest on the exact same train/test split from Section 2, then scores all three (including the Week-4 rule-based baseline) with Precision, Recall, and F1. AUC-ROC is computed for the two real models only — the baseline outputs a hard 0/1 flag, not a probability, so an AUC score for it would not be meaningful.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_proba = lr.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

results = pd.DataFrame({
    'model': ['Baseline (rule)', 'Logistic Regression', 'Random Forest'],
    'precision': [precision_score(y_test, baseline_test), precision_score(y_test, lr_pred), precision_score(y_test, rf_pred)],
    'recall':    [recall_score(y_test, baseline_test), recall_score(y_test, lr_pred), recall_score(y_test, rf_pred)],
    'f1':        [f1_score(y_test, baseline_test), f1_score(y_test, lr_pred), f1_score(y_test, rf_pred)],
    'auc_roc':   [None, roc_auc_score(y_test, lr_proba), roc_auc_score(y_test, rf_proba)],
})
print(results.to_string(index=False))


**Fill this in after running the cell above — do not guess the numbers:**

- Which model had the best Precision / Recall / F1: `[FILL IN FROM PRINTED TABLE]`
- Did either real model beat the baseline's F1, and by how much: `[FILL IN]`
- Logistic Regression AUC-ROC: `[FILL IN]` · Random Forest AUC-ROC: `[FILL IN]`
- One or two sentences on what this means for the baseline built in Week 4 (did the extra modeling effort pay off, or was the simple rule already close to as good): `[FILL IN]`

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The code below prints Logistic Regression's coefficients and Random Forest's permutation importances side by side, then pulls a handful of real false positives and false negatives to look for patterns.

In [ ]:
from sklearn.inspection import permutation_importance

lr_coefs = pd.DataFrame({'feature': X_train.columns, 'coefficient': lr.coef_[0]})
lr_coefs = lr_coefs.reindex(lr_coefs.coefficient.abs().sort_values(ascending=False).index)
print("--- Logistic Regression coefficients (top 10 by magnitude) ---")
print(lr_coefs.head(10).to_string(index=False))

perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
rf_importance = pd.DataFrame({'feature': X_train.columns, 'importance': perm.importances_mean}).sort_values('importance', ascending=False)
print("\n--- Random Forest permutation importance (top 10) ---")
print(rf_importance.head(10).to_string(index=False))

test_df = model_df.iloc[test_idx].copy()
test_df['rf_pred'] = rf_pred
test_df['true_label'] = y_test.values
false_positives = test_df[(test_df['rf_pred'] == 1) & (test_df['true_label'] == 0)]
false_negatives = test_df[(test_df['rf_pred'] == 0) & (test_df['true_label'] == 1)]
print(f"\nFalse positives: {len(false_positives)}, False negatives: {len(false_negatives)}")
print(false_positives[['content_hash_id', 'imp_prev30', 'ctr_prev30', 'days_since_last_update']].head(5).to_string(index=False))
print(false_negatives[['content_hash_id', 'imp_prev30', 'ctr_prev30', 'days_since_last_update']].head(5).to_string(index=False))


**Fill this in after running the cell above:**

- Do Logistic Regression's top coefficients and Random Forest's top permutation importances agree on the same 2-3 features? `[FILL IN — this agreement/disagreement is the main finding of this whole lane]`
- What do the printed false positives have in common, if anything (e.g. low `imp_prev30`)? `[FILL IN]`
- What do the printed false negatives have in common, if anything (e.g. `days_since_last_update` near the 90-day threshold)? `[FILL IN]`

**Careful words:** this analysis shows which signals the model leans on and how they associate with decline — it does not prove any single signal causes decline, and `is_declining_label` remains a proxy based on the current window, not a guaranteed future outcome.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it (Sections 3 and 4's markdown have `[FILL IN]` placeholders — complete these with real printed numbers before checking this box)
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.